# Plane integral of charge density alon z
## We assume the cube file is in a orthogonal cell (a,b,c)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import physical_constants
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
bohr_to_ang = physical_constants["Bohr radius"][0] * 1e10  # Å

# path to cube file to be displayed

In [2]:
import numpy as np
from scipy.ndimage import map_coordinates


# ----------------------------
# internal helpers
# ----------------------------

def _normalize_field_type(field_type):
    """
    Normalize user-provided field type aliases.
    Accepted families:
      - charge
      - hartree
      - rydberg
    """
    key = str(field_type).strip().lower()

    aliases = {
        "charge": "charge",
        "rho": "charge",
        "density": "charge",
        "charge_density": "charge",

        "hartree": "hartree",
        "ha": "hartree",
        "h": "hartree",
        "potential_hartree": "hartree",

        "rydberg": "rydberg",
        "ry": "rydberg",
        "potential_rydberg": "rydberg",
    }

    if key not in aliases:
        raise ValueError(
            f"Unknown field_type={field_type!r}. "
            "Use one of: 'charge', 'hartree', 'rydberg'."
        )
    return aliases[key]


def _cube_geometry(header_lines, bohr_to_ang):
    """
    Parse cube header geometry.

    Returns a dictionary with:
      - origin_ang: origin in Å
      - step_matrix_ang: 3x3 matrix, columns are voxel step vectors in Å
      - cell_matrix_ang: 3x3 matrix, columns are full cell vectors in Å
      - nxyz: grid shape [nx, ny, nz]
      - inv_step_matrix_ang: inverse of step_matrix_ang
    """
    origin_tokens = header_lines[2].split()
    xgrid_tokens = header_lines[3].split()
    ygrid_tokens = header_lines[4].split()
    zgrid_tokens = header_lines[5].split()

    # cube convention: line 3 is [natoms, ox, oy, oz]
    origin_bohr = np.array(list(map(float, origin_tokens[1:4])), dtype=float)

    nx = abs(int(xgrid_tokens[0]))
    ny = abs(int(ygrid_tokens[0]))
    nz = abs(int(zgrid_tokens[0]))

    vx_bohr = np.array(list(map(float, xgrid_tokens[1:4])), dtype=float)
    vy_bohr = np.array(list(map(float, ygrid_tokens[1:4])), dtype=float)
    vz_bohr = np.array(list(map(float, zgrid_tokens[1:4])), dtype=float)

    origin_ang = origin_bohr * bohr_to_ang
    step_matrix_ang = bohr_to_ang * np.column_stack([vx_bohr, vy_bohr, vz_bohr])

    nxyz = np.array([nx, ny, nz], dtype=int)
    cell_matrix_ang = step_matrix_ang @ np.diag(nxyz)

    return {
        "origin_ang": origin_ang,
        "step_matrix_ang": step_matrix_ang,
        "cell_matrix_ang": cell_matrix_ang,
        "inv_step_matrix_ang": np.linalg.inv(step_matrix_ang),
        "nxyz": nxyz,
    }


def _convert_cube_values(values, field_type, bohr_to_ang):
    """
    Convert cube values to requested physical units.

    charge:
        e / bohr^3  ->  e / Å^3
    hartree:
        unchanged
    rydberg:
        unchanged
    """
    field_type = _normalize_field_type(field_type)
    values = np.asarray(values, dtype=float)

    if field_type == "charge":
        out = values / (bohr_to_ang ** 3)
        unit = "e/Å³"
        quantity_name = "Charge density"
    elif field_type == "hartree":
        out = values.copy()
        unit = "Hartree"
        quantity_name = "Potential"
    elif field_type == "rydberg":
        out = values.copy()
        unit = "Rydberg"
        quantity_name = "Potential"
    else:
        raise RuntimeError("Unexpected field type after normalization.")

    return out, unit, quantity_name, field_type


def _orthonormal_plane_basis(direction):
    """
    Given a nonzero direction vector, return:
      - uhat: unit vector along direction
      - e1, e2: orthonormal basis spanning the plane perpendicular to uhat
    """
    direction = np.asarray(direction, dtype=float)
    norm = np.linalg.norm(direction)
    if norm == 0:
        raise ValueError("P1 and P2 must be different points.")

    uhat = direction / norm

    # choose a Cartesian axis least parallel to uhat
    cart_axes = np.eye(3)
    ref = cart_axes[np.argmin(np.abs(cart_axes @ uhat))]

    e1 = ref - np.dot(ref, uhat) * uhat
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(uhat, e1)
    e2 /= np.linalg.norm(e2)

    return uhat, e1, e2


def _default_perp_rectangle(cell_matrix_ang, e1, e2):
    """
    Default rectangle sizes in the plane perpendicular to the analysis direction.

    We use the bounding rectangle of the projection of the full periodic cell
    parallelepiped onto the perpendicular plane basis (e1, e2).
    """
    frac_corners = np.array(
        [
            [0, 0, 0],
            [1, 0, 0],
            [0, 1, 0],
            [0, 0, 1],
            [1, 1, 0],
            [1, 0, 1],
            [0, 1, 1],
            [1, 1, 1],
        ],
        dtype=float,
    )

    # columns of cell_matrix_ang are the three cell vectors
    corners = frac_corners @ cell_matrix_ang.T

    u_proj = corners @ e1
    v_proj = corners @ e2

    L_default = u_proj.max() - u_proj.min()
    W_default = v_proj.max() - v_proj.min()

    return float(L_default), float(W_default)


def _centered_axis(length, target_step):
    """
    Build 1D centered sampling coordinates across an interval of size `length`.

    Returns:
      coords  : cell-centered coordinates from -length/2 to +length/2
      step    : exact step actually used
      npts    : number of points
    """
    length = float(length)
    target_step = float(target_step)

    if length <= 0:
        raise ValueError("A rectangle length/width must be > 0.")
    if target_step <= 0:
        raise ValueError("Sampling step must be > 0.")

    npts = max(1, int(np.ceil(length / target_step)))
    step = length / npts
    coords = -0.5 * length + (np.arange(npts) + 0.5) * step

    return coords, step, npts


def _line_centers(total_length, target_dl):
    """
    Build line-centered coordinates from P1 to P2.

    Returns:
      l_centers : line bin centers
      l_edges   : line bin edges
      dl        : exact step actually used
      npts      : number of line bins
    """
    total_length = float(total_length)
    target_dl = float(target_dl)

    if total_length <= 0:
        raise ValueError("The line length |P2-P1| must be > 0.")
    if target_dl <= 0:
        raise ValueError("dl must be > 0.")

    npts = max(1, int(np.ceil(total_length / target_dl)))
    dl = total_length / npts
    l_edges = np.linspace(0.0, total_length, npts + 1)
    l_centers = 0.5 * (l_edges[:-1] + l_edges[1:])

    return l_centers, l_edges, dl, npts


def _cartesian_to_grid_indices(points_ang, geom):
    """
    Convert Cartesian points in Å to cube grid coordinates (i, j, k),
    where the field is sampled as rho[i, j, k].

    `points_ang` shape can be (..., 3).
    """
    pts = np.asarray(points_ang, dtype=float)
    pts_2d = pts.reshape(-1, 3)

    rel = pts_2d - geom["origin_ang"]
    ijk = rel @ geom["inv_step_matrix_ang"].T

    return ijk.reshape(pts.shape)


def _sample_cube_periodic(points_ang, values_phys, geom, order=1):
    """
    Periodic interpolation of the cube field at arbitrary Cartesian points.

    Uses scipy.ndimage.map_coordinates with mode='wrap'.
    """
    if order not in (0, 1, 2, 3, 4, 5):
        raise ValueError("Interpolation order must be an integer between 0 and 5.")

    ijk = _cartesian_to_grid_indices(points_ang, geom).reshape(-1, 3)
    coords = np.vstack([ijk[:, 0], ijk[:, 1], ijk[:, 2]])

    sampled = map_coordinates(
        values_phys,
        coords,
        order=order,
        mode="wrap",
        prefilter=(order > 1),
    )

    return sampled.reshape(np.asarray(points_ang).shape[:-1])


# ----------------------------
# public functions
# ----------------------------

def cube_plane_average_profile(
    header_lines,
    cube_values,
    bohr_to_ang,
    P1,
    P2,
    field_type="charge",
    dl=0.1,
    L=None,
    W=None,
    du=0.1,
    dv=None,
    order=1,
):
    """
    1D profile along the line P1 -> P2 from averages over perpendicular rectangles.

    Parameters
    ----------
    header_lines : list[str]
        Cube header lines.
    cube_values : ndarray, shape (nx, ny, nz)
        Scalar field read from cube.
    bohr_to_ang : float
        Conversion factor Bohr -> Å.
    P1, P2 : array-like, shape (3,)
        Endpoints in Å.
    field_type : {'charge', 'hartree', 'rydberg'}
        Type of scalar field.
    dl : float, default 0.1
        Target step along P1 -> P2 in Å.
    L, W : float or None
        Rectangle size in the plane perpendicular to P2-P1, in Å.
        If None, use the default largest projected cross-section of the cell.
    du, dv : float
        Target in-plane sampling steps in Å.
    order : int
        Interpolation order for scipy.ndimage.map_coordinates.

    Returns
    -------
    result : dict
        Contains profile, geometry, sampling metadata, labels, and units.
    """
    geom = _cube_geometry(header_lines, bohr_to_ang)
    values_phys, unit, quantity_name, field_type = _convert_cube_values(
        cube_values, field_type, bohr_to_ang
    )

    P1 = np.asarray(P1, dtype=float)
    P2 = np.asarray(P2, dtype=float)
    direction = P2 - P1
    axis_length = np.linalg.norm(direction)

    uhat, e1, e2 = _orthonormal_plane_basis(direction)
    L_default, W_default = _default_perp_rectangle(geom["cell_matrix_ang"], e1, e2)

    if L is None:
        L = L_default
    if W is None:
        W = W_default
    if dv is None:
        dv = du

    l_ang, l_edges_ang, dl_eff, nl = _line_centers(axis_length, dl)
    u_ang, du_eff, nu = _centered_axis(L, du)
    v_ang, dv_eff, nv = _centered_axis(W, dv)

    U, V = np.meshgrid(u_ang, v_ang, indexing="ij")
    plane_offsets = U[..., None] * e1 + V[..., None] * e2
    plane_offsets_flat = plane_offsets.reshape(-1, 3)

    profile = np.empty(nl, dtype=float)

    for i, s in enumerate(l_ang):
        center = P1 + s * uhat
        points = center[None, :] + plane_offsets_flat
        vals = _sample_cube_periodic(points, values_phys, geom, order=order)
        profile[i] = vals.mean()

    rectangle_area = float(L * W)
    dA = float(du_eff * dv_eff)
    dV = float(dA * dl_eff)

    direction_str = f"({direction[0]:.6g}, {direction[1]:.6g}, {direction[2]:.6g})"
    xlabel = f"L (Å) along {direction_str}"

    if field_type == "charge":
        ylabel = f"Average charge density ({unit})"
    else:
        ylabel = f"Average potential ({unit})"

    return {
        # main data
        "l_ang": l_ang,
        "l_edges_ang": l_edges_ang,
        "profile": profile,

        # labels / units
        "xlabel": xlabel,
        "ylabel": ylabel,
        "field_unit": unit,
        "field_type": field_type,
        "quantity_name": quantity_name,

        # geometry
        "P1_ang": P1,
        "P2_ang": P2,
        "direction_ang": direction,
        "axis_length_ang": axis_length,
        "axis_hat": uhat,
        "plane_e1": e1,
        "plane_e2": e2,

        # rectangle
        "L_ang": float(L),
        "W_ang": float(W),
        "L_default_ang": float(L_default),
        "W_default_ang": float(W_default),
        "rectangle_area_ang2": rectangle_area,

        # plane sampling
        "u_ang": u_ang,
        "v_ang": v_ang,
        "du_ang": float(du_eff),
        "dv_ang": float(dv_eff),
        "dA_ang2": dA,
        "nu": int(nu),
        "nv": int(nv),

        # line sampling
        "dl_ang": float(dl_eff),
        "dV_ang3": dV,
        "nl": int(nl),

        # useful references
        "origin_ang": geom["origin_ang"],
        "cell_matrix_ang": geom["cell_matrix_ang"],
        "step_matrix_ang": geom["step_matrix_ang"],
        "grid_shape": geom["nxyz"],
    }


def cube_perpendicular_plane_map(
    header_lines,
    cube_values,
    bohr_to_ang,
    P1,
    P2,
    position=0.0,
    field_type="charge",
    L=None,
    W=None,
    du=0.1,
    dv=None,
    sigma=0.0,
    dn=None,
    truncate=4.0,
    order=1,
):
    """
    2D map in the plane perpendicular to P1 -> P2 at a given position, with
    optional Gaussian broadening along the normal direction.

    Parameters
    ----------
    header_lines : list[str]
        Cube header lines.
    cube_values : ndarray, shape (nx, ny, nz)
        Scalar field read from cube.
    bohr_to_ang : float
        Conversion factor Bohr -> Å.
    P1, P2 : array-like, shape (3,)
        Endpoints in Å defining the axis.
    position : float, default 0.0
        Position along P1 -> P2 in Å. position=0 means plane through P1.
    field_type : {'charge', 'hartree', 'rydberg'}
        Type of scalar field.
    L, W : float or None
        Rectangle size in the perpendicular plane, in Å.
        If None, use default largest projected cross-section.
    du, dv : float
        Target in-plane sampling steps in Å.
    sigma : float, default 0.0
        Gaussian broadening sigma in Å along the plane normal.
        sigma=0 means direct slice.
    dn : float or None
        Sampling step along the normal direction for the Gaussian quadrature.
        If None, choose an automatic value.
    truncate : float
        Gaussian cutoff in units of sigma.
    order : int
        Interpolation order for scipy.ndimage.map_coordinates.

    Returns
    -------
    result : dict
        Contains 2D map, geometry, sampling metadata, labels, and units.
    """
    geom = _cube_geometry(header_lines, bohr_to_ang)
    values_phys, unit, quantity_name, field_type = _convert_cube_values(
        cube_values, field_type, bohr_to_ang
    )

    P1 = np.asarray(P1, dtype=float)
    P2 = np.asarray(P2, dtype=float)
    direction = P2 - P1
    axis_length = np.linalg.norm(direction)

    uhat, e1, e2 = _orthonormal_plane_basis(direction)
    L_default, W_default = _default_perp_rectangle(geom["cell_matrix_ang"], e1, e2)

    if L is None:
        L = L_default
    if W is None:
        W = W_default
    if dv is None:
        dv = du

    u_ang, du_eff, nu = _centered_axis(L, du)
    v_ang, dv_eff, nv = _centered_axis(W, dv)

    U, V = np.meshgrid(u_ang, v_ang, indexing="ij")
    center = P1 + float(position) * uhat

    plane_points = center + U[..., None] * e1 + V[..., None] * e2
    plane_points_flat = plane_points.reshape(-1, 3)

    if sigma < 0:
        raise ValueError("sigma must be >= 0.")

    if sigma == 0:
        field_2d = _sample_cube_periodic(plane_points_flat, values_phys, geom, order=order)
        field_2d = field_2d.reshape(U.shape)
        normal_offsets = np.array([0.0])
        normal_weights = np.array([1.0])
        dn_eff = 0.0
    else:
        if dn is None:
            # reasonable automatic normal step
            dn = min(du_eff, dv_eff, max(sigma / 3.0, 0.05))
        if dn <= 0:
            raise ValueError("dn must be > 0 when sigma > 0.")
        if truncate <= 0:
            raise ValueError("truncate must be > 0.")

        n_side = max(1, int(np.ceil(truncate * sigma / dn)))
        normal_offsets = np.arange(-n_side, n_side + 1, dtype=float) * dn
        normal_weights = np.exp(-0.5 * (normal_offsets / sigma) ** 2)
        normal_weights /= normal_weights.sum()
        dn_eff = float(dn)

        field_2d = np.zeros(U.shape, dtype=float)
        for t, w in zip(normal_offsets, normal_weights):
            vals = _sample_cube_periodic(
                plane_points_flat + t * uhat,
                values_phys,
                geom,
                order=order,
            ).reshape(U.shape)
            field_2d += w * vals

    if field_type == "charge":
        colorbar_label = f"Charge density ({unit})"
    else:
        colorbar_label = f"Potential ({unit})"

    return {
        # main data
        "u_ang_grid": U,
        "v_ang_grid": V,
        "map_2d": field_2d,

        # labels / units
        "xlabel": "u (Å)",
        "ylabel": "v (Å)",
        "colorbar_label": colorbar_label,
        "field_unit": unit,
        "field_type": field_type,
        "quantity_name": quantity_name,

        # geometry
        "P1_ang": P1,
        "P2_ang": P2,
        "direction_ang": direction,
        "axis_length_ang": axis_length,
        "axis_hat": uhat,
        "plane_e1": e1,
        "plane_e2": e2,
        "plane_center_ang": center,
        "position_ang": float(position),

        # rectangle
        "L_ang": float(L),
        "W_ang": float(W),
        "L_default_ang": float(L_default),
        "W_default_ang": float(W_default),

        # plane sampling
        "u_ang": u_ang,
        "v_ang": v_ang,
        "du_ang": float(du_eff),
        "dv_ang": float(dv_eff),
        "dA_ang2": float(du_eff * dv_eff),
        "nu": int(nu),
        "nv": int(nv),

        # Gaussian broadening
        "sigma_ang": float(sigma),
        "dn_ang": float(dn_eff),
        "normal_offsets_ang": normal_offsets,
        "normal_weights": normal_weights,
        "truncate": float(truncate),

        # useful references
        "origin_ang": geom["origin_ang"],
        "cell_matrix_ang": geom["cell_matrix_ang"],
        "step_matrix_ang": geom["step_matrix_ang"],
        "grid_shape": geom["nxyz"],
    }

In [3]:
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


def read_cube_full(filename):
    """
    Read a Gaussian cube file.

    Returns
    -------
    header_lines : list[str]
        Original header lines up to atom list (used for writing)
    atom_lines : list[str]
        Atom specification lines
    rho : ndarray (nx, ny, nz)
        Volumetric data as stored in the cube file
    grid_shape : tuple
        (nx, ny, nz)
    """
    filename = str(filename)

    with open(filename, "r") as f:
        lines = f.readlines()

    header_lines = lines[:2]

    # Cube line 3: natoms origin_x origin_y origin_z
    natoms = abs(int(lines[2].split()[0]))
    grid_lines = lines[2:6]
    header_lines += grid_lines

    nx = abs(int(grid_lines[1].split()[0]))
    ny = abs(int(grid_lines[2].split()[0]))
    nz = abs(int(grid_lines[3].split()[0]))

    atom_lines = lines[6:6 + natoms]
    data_lines = lines[6 + natoms:]

    raw = []
    for line in data_lines:
        raw.extend(map(float, line.split()))

    expected = nx * ny * nz
    if len(raw) != expected:
        raise ValueError(
            f"Cube data size mismatch in {filename}: "
            f"expected {expected} values, found {len(raw)}."
        )

    rho = np.array(raw, dtype=float).reshape((nx, ny, nz))

    return header_lines, atom_lines, rho, (nx, ny, nz)


def _parse_point(text):
    """
    Parse a 3D point from flexible text formats such as:
      '0.1 1 3'
      '0.1, 1, 3'
      '0.1 1 ; 3'
      '0.1 ; 1 ; 3'
    """
    if text is None:
        raise ValueError("Point text is empty.")

    s = str(text).strip()
    if not s:
        raise ValueError("Point text is empty.")

    # Extract numbers robustly, including scientific notation
    nums = re.findall(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?', s)

    if len(nums) != 3:
        raise ValueError(
            f"Could not parse 3 coordinates from {text!r}. "
            "Examples: '0.1 1 3' or '0.1, 1, 3'."
        )

    return np.array([float(x) for x in nums], dtype=float)


def _float_or_none(text):
    s = str(text).strip()
    return None if s == "" else float(s)


def _collect_cube_files(download_dir=Path.home() / "opt" / "useful-notebooks" / "CubeFiles" / "Downloads"):
    patterns = ["*.cube", "*.CUBE", "*.cub", "*.CUB"]
    files = []
    for patt in patterns:
        files.extend(download_dir.glob(patt))
    files = sorted(set(files), key=lambda p: p.name.lower())
    return files


# ----------------------------
# widgets
# ----------------------------

cube_files = _collect_cube_files()

if cube_files:
    cube_file_dropdown = widgets.Dropdown(
        options=[(p.name, str(p)) for p in cube_files],
        description="Cube file:",
        layout=widgets.Layout(width="700px"),
        style={"description_width": "120px"},
    )
else:
    cube_file_dropdown = widgets.Dropdown(
        options=[("No .cube/.cub files found in ./Downloads", "")],
        description="Cube file:",
        layout=widgets.Layout(width="700px"),
        style={"description_width": "120px"},
        disabled=True,
    )

field_type_dropdown = widgets.Dropdown(
    options=[
        ("charge", "charge"),
        ("hartree", "hartree"),
        ("rydberg", "rydberg"),
    ],
    value="charge",
    description="Field type:",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "120px"},
)

plot_type_dropdown = widgets.Dropdown(
    options=[
        ("1D line plot", "1d"),
        ("2D plot", "2d"),
    ],
    value="1d",
    description="Plot type:",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "120px"},
)

P1_text = widgets.Text(
    value="0 0 0",
    description="P1 (Å):",
    placeholder="e.g. 0 0 0",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "120px"},
)

P2_text = widgets.Text(
    value="0 0 10",
    description="P2 (Å):",
    placeholder="e.g. 0 0 10",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "120px"},
)

L_text = widgets.Text(
    value="",
    description="L (Å):",
    placeholder="empty = use default",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

W_text = widgets.Text(
    value="",
    description="W (Å):",
    placeholder="empty = use default",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

dl_text = widgets.Text(
    value="0.1",
    description="dl (Å):",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

P0_text = widgets.Text(
    value="",
    description="P0 (Å):",
    placeholder="empty = midpoint between P1 and P2",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "120px"},
)

interp_order = widgets.BoundedIntText(
    value=1,
    min=0,
    max=5,
    step=1,
    description="Interp. order:",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

sigma_text = widgets.Text(
    value="0.1",
    description="sigma (Å):",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

dn_text = widgets.Text(
    value="3",
    description="dn (Å):",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "120px"},
)

plot_button = widgets.Button(
    description="Plot",
    button_style="primary",
    layout=widgets.Layout(width="120px"),
)

message_html = widgets.HTML()
output = widgets.Output()

row1 = widgets.HBox([cube_file_dropdown])
row2 = widgets.HBox([field_type_dropdown, plot_type_dropdown])
row3 = widgets.HBox([P1_text, P2_text])
row4 = widgets.HBox([L_text, W_text])
row5_1d = widgets.HBox([dl_text, interp_order])
row5_2d = widgets.HBox([P0_text, interp_order])
row6_2d = widgets.HBox([sigma_text, dn_text])


def _update_visibility(*args):
    if plot_type_dropdown.value == "1d":
        dl_text.layout.display = ""
        P0_text.layout.display = "none"
        sigma_text.layout.display = "none"
        dn_text.layout.display = "none"

        row5_1d.layout.display = ""
        row5_2d.layout.display = "none"
        row6_2d.layout.display = "none"
    else:
        dl_text.layout.display = "none"
        P0_text.layout.display = ""
        sigma_text.layout.display = ""
        dn_text.layout.display = ""

        row5_1d.layout.display = "none"
        row5_2d.layout.display = ""
        row6_2d.layout.display = ""


plot_type_dropdown.observe(_update_visibility, names="value")
_update_visibility()


def _on_plot_clicked(_):
    with output:
        clear_output()
        message_html.value = ""

        try:
            if not cube_file_dropdown.value:
                raise ValueError("No cube file is available in ./Downloads")

            filename = cube_file_dropdown.value
            field_type = field_type_dropdown.value
            plot_type = plot_type_dropdown.value
            order = int(interp_order.value)

            P1 = _parse_point(P1_text.value)
            P2 = _parse_point(P2_text.value)

            if np.allclose(P1, P2):
                raise ValueError("P1 and P2 must be different.")

            L_val = _float_or_none(L_text.value)
            W_val = _float_or_none(W_text.value)

            header_lines, atom_lines, cube_values, grid_shape = read_cube_full(filename)

            if plot_type == "1d":
                dl_val = float(dl_text.value)

                result = cube_plane_average_profile(
                    header_lines=header_lines,
                    cube_values=cube_values,
                    bohr_to_ang=bohr_to_ang,
                    P1=P1,
                    P2=P2,
                    field_type=field_type,
                    dl=dl_val,
                    L=L_val,
                    W=W_val,
                    order=order,
                )

                # populate defaults only if fields were empty
                if L_val is None:
                    L_text.value = f"{result['L_default_ang']:.6g}"
                if W_val is None:
                    W_text.value = f"{result['W_default_ang']:.6g}"

                fig, ax = plt.subplots(figsize=(7, 4.5))
                ax.plot(result["l_ang"], result["profile"])
                ax.set_xlabel(result["xlabel"])
                ax.set_ylabel(result["ylabel"])
                ax.set_title(Path(filename).name)
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

                print(f"Grid shape: {grid_shape}")
                print(f"L used = {result['L_ang']:.6g} Å")
                print(f"W used = {result['W_ang']:.6g} Å")
                print(f"Rectangle area = {result['rectangle_area_ang2']:.6g} Å²")
                print(f"dl used = {result['dl_ang']:.6g} Å")

            else:
                axis_length = np.linalg.norm(P2 - P1)
                P0_val = _float_or_none(P0_text.value)
                if P0_val is None:
                    P0_val = 0.5 * axis_length
                    P0_text.value = f"{P0_val:.6g}"

                sigma_val = float(sigma_text.value)
                dn_val = float(dn_text.value)

                result = cube_perpendicular_plane_map(
                    header_lines=header_lines,
                    cube_values=cube_values,
                    bohr_to_ang=bohr_to_ang,
                    P1=P1,
                    P2=P2,
                    position=P0_val,
                    field_type=field_type,
                    L=L_val,
                    W=W_val,
                    sigma=sigma_val,
                    dn=dn_val,
                    order=order,
                )

                # populate defaults only if fields were empty
                if L_val is None:
                    L_text.value = f"{result['L_default_ang']:.6g}"
                if W_val is None:
                    W_text.value = f"{result['W_default_ang']:.6g}"

                fig, ax = plt.subplots(figsize=(6.5, 5.5))
                pcm = ax.pcolormesh(
                    result["u_ang_grid"],
                    result["v_ang_grid"],
                    result["map_2d"],
                    shading="auto",
                )
                cbar = plt.colorbar(pcm, ax=ax)
                cbar.set_label(result["colorbar_label"])

                ax.set_xlabel(result["xlabel"])
                ax.set_ylabel(result["ylabel"])
                ax.set_title(f"{Path(filename).name}   |   P0 = {result['position_ang']:.6g} Å")
                ax.set_aspect("equal")
                plt.tight_layout()
                plt.show()

                print(f"Grid shape: {grid_shape}")
                print(f"L used = {result['L_ang']:.6g} Å")
                print(f"W used = {result['W_ang']:.6g} Å")
                print(f"P0 used = {result['position_ang']:.6g} Å")
                print(f"sigma used = {result['sigma_ang']:.6g} Å")
                print(f"dn used = {result['dn_ang']:.6g} Å")

        except Exception as exc:
            print(f"Error: {exc}")


plot_button.on_click(_on_plot_clicked)

ui = widgets.VBox([
    row1,
    row2,
    row3,
    row4,
    row5_1d,
    row5_2d,
    row6_2d,
    plot_button,
    message_html,
    output,
])

display(ui)